# IEEE-CIS Fraud Detection — Modelling

## Goal

Train three gradient boosting models on the engineered features from notebook 02.
Each model is logged as a separate MLflow run so results are comparable.

## Why these three models?

| Model | Strength |
|---|---|
| XGBoost | Battle-tested, great default performance, wide community support |
| LightGBM | Fastest training, handles high cardinality well, good on large datasets |
| CatBoost | Best native categorical handling, less hyperparameter tuning needed |

All three are gradient boosting — ensemble of decision trees built sequentially,
each tree correcting the errors of the previous one.

## Evaluation Metric

**Primary: AUC-PR (Area Under Precision-Recall Curve)**

Not accuracy. Not AUC-ROC. AUC-PR because:
- 96.5% accuracy is achievable by predicting everything as legitimate — useless
- AUC-ROC is optimistic under class imbalance (28:1 ratio)
- AUC-PR focuses on the positive (fraud) class — more informative when fraud is rare

**Secondary: AUC-ROC, F1** — for completeness and comparison

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report
)

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

# Load processed data
train_df = pd.read_parquet('data/processed/train_features.parquet')
val_df   = pd.read_parquet('data/processed/val_features.parquet')

with open('data/processed/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols]
y_train = train_df['isFraud']
X_val   = val_df[feature_cols]
y_val   = val_df['isFraud']

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'Val fraud rate:   {y_val.mean()*100:.2f}%')

X_train: (442905, 443)
X_val:   (147635, 443)
Features: 443
Train fraud rate: 3.51%
Val fraud rate:   3.45%


In [11]:
from sklearn.preprocessing import LabelEncoder

# Label encode remaining object columns — fit on train only to prevent leakage
obj_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f'Object columns to encode: {obj_cols}')

for col in obj_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col]   = le.transform(X_val[col].astype(str).map(
        lambda x: x if x in le.classes_ else le.classes_[0]
    ))

print('Done. All columns are now numeric.')
print(X_train.dtypes.value_counts())

Object columns to encode: ['P_emaildomain', 'R_emaildomain', 'M4', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'card4_6']
Done. All columns are now numeric.
float64    402
int64       41
Name: count, dtype: int64


## Chapter 1 — Class Imbalance Strategy

28:1 ratio means the model will see 28 legitimate transactions for every 1 fraud.
Without correction it will learn to predict everything as legitimate.

**Solution: `scale_pos_weight`**

Tells the model to penalise missed fraud more heavily by upweighting the positive class:

```
scale_pos_weight = count(negative) / count(positive)
                 = 569,877 / 20,663
                 ≈ 27.6
```

This makes the model treat each fraud case as if it were 27.6 legitimate cases —
balancing the gradient updates during training.

In [12]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

scale_pos_weight: 27.46


## Chapter 2 — MLflow Setup

Each model is logged as a separate MLflow run under one experiment.
This lets you compare AUC-PR, AUC-ROC, F1 side by side in the MLflow UI.

We log:
- All hyperparameters
- AUC-PR, AUC-ROC, F1 scores
- Confusion matrix as an artifact
- Feature importance plot as an artifact
- The model itself (for later deployment)

In [13]:
mlflow.set_experiment('fraud-detection')
print('MLflow experiment set: fraud-detection')

MLflow experiment set: fraud-detection


## Chapter 3 — XGBoost

Start with XGBoost — most familiar, good baseline.

Key parameters:
- `scale_pos_weight` — handles class imbalance
- `eval_metric='aucpr'` — optimises for AUC-PR during training
- `early_stopping_rounds=50` — stops if val AUC-PR doesn't improve for 50 rounds
- `n_estimators=1000` — high ceiling, early stopping will find the right number

### Your task

Fill in the blanks and run the XGBoost training cell below.

In [14]:
import joblib, os
from xgboost import XGBClassifier

os.makedirs('models', exist_ok=True)

xgb_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'aucpr',
    'early_stopping_rounds': 50,
    'random_state': 42,
    'n_jobs': -1
}

with mlflow.start_run(run_name='xgboost_baseline'):
    mlflow.log_params(xgb_params)
    
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    
    joblib.dump(xgb_model, 'models/xgboost_baseline.pkl')
    
    print(f'XGBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')
    print(f'  Best iteration: {xgb_model.best_iteration}')

[0]	validation_0-aucpr:0.30385
[100]	validation_0-aucpr:0.45795
[200]	validation_0-aucpr:0.48388
[300]	validation_0-aucpr:0.49758
[400]	validation_0-aucpr:0.50510
[500]	validation_0-aucpr:0.50944
[600]	validation_0-aucpr:0.51579
[700]	validation_0-aucpr:0.51976
[800]	validation_0-aucpr:0.52250
[830]	validation_0-aucpr:0.52237
XGBoost Results:
  AUC-PR:  0.5231
  AUC-ROC: 0.9017
  F1:      0.4017
  Best iteration: 780
🏃 View run xgboost_baseline at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo/#/experiments/51a2010c-aa0f-4cbe-a06b-ae4cdfbda8ca/runs/18faf7d7-690c-4afc-8713-90add13f3fa6
🧪 View experiment at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo/#/experiments/51a2010c-aa0f-4cbe

## Chapter 4 — LightGBM

LightGBM is faster than XGBoost on large datasets — uses leaf-wise tree growth
instead of level-wise, finding better splits faster.

Key difference from XGBoost:
- `is_unbalance=True` instead of `scale_pos_weight` — automatic balancing
- `metric='average_precision'` — equivalent to AUC-PR

In [18]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'is_unbalance': True,
    'metric': 'average_precision',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

with mlflow.start_run(run_name='lightgbm_baseline'):
    mlflow.log_params(lgbm_params)
    
    lgbm_model = LGBMClassifier(**lgbm_params)
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50), log_evaluation(100)]
    )
    
    y_pred_proba = lgbm_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    
    joblib.dump(lgbm_model, 'models/lightgbm_baseline.pkl')
    
    print(f'LightGBM Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')
    print(f'  Best iteration: {lgbm_model.best_iteration_}')


Training until validation scores don't improve for 50 rounds
[100]	valid_0's average_precision: 0.4618
[200]	valid_0's average_precision: 0.487306
[300]	valid_0's average_precision: 0.502486
[400]	valid_0's average_precision: 0.509228
[500]	valid_0's average_precision: 0.515861
[600]	valid_0's average_precision: 0.521164
[700]	valid_0's average_precision: 0.526243
Early stopping, best iteration is:
[705]	valid_0's average_precision: 0.526545
LightGBM Results:
  AUC-PR:  0.5265
  AUC-ROC: 0.9057
  F1:      0.3784
  Best iteration: 705
🏃 View run lightgbm_baseline at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo/#/experiments/51a2010c-aa0f-4cbe-a06b-ae4cdfbda8ca/runs/e3033e5e-4d07-4005-ac54-804808430d94
🧪 View experiment at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourc

## Chapter 5 — CatBoost

CatBoost handles categorical features natively — no label encoding needed.
It builds symmetric trees and uses ordered boosting to reduce overfitting.

Key difference:
- `auto_class_weights='Balanced'` — automatic class weight calculation
- `cat_features` — pass categorical column indices directly, no encoding needed
- `eval_metric='PRAUC'` — AUC-PR in CatBoost notation

In [20]:
from catboost import CatBoostClassifier

catboost_params = {
    'iterations': 1000,
    'depth': 6,
    'learning_rate': 0.05,
    'auto_class_weights': 'Balanced',
    'eval_metric': 'PRAUC',
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 100
}

with mlflow.start_run(run_name='catboost_baseline'):
    mlflow.log_params(catboost_params)
    
    cat_model = CatBoostClassifier(**catboost_params)
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val)
    )
    
    y_pred_proba = cat_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    joblib.dump(cat_model, 'models/catboost_baseline.pkl')
    
    print(f'CatBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')

0:	learn: 0.8404478	test: 0.8258005	best: 0.8258005 (0)	total: 283ms	remaining: 4m 42s
100:	learn: 0.9091908	test: 0.8885900	best: 0.8885900 (100)	total: 30.2s	remaining: 4m 28s
200:	learn: 0.9226816	test: 0.8972655	best: 0.8972655 (200)	total: 59.4s	remaining: 3m 55s
300:	learn: 0.9336059	test: 0.9038322	best: 0.9038322 (300)	total: 1m 26s	remaining: 3m 20s
400:	learn: 0.9435038	test: 0.9091574	best: 0.9091819 (399)	total: 1m 55s	remaining: 2m 51s
500:	learn: 0.9502690	test: 0.9114804	best: 0.9114804 (500)	total: 2m 23s	remaining: 2m 23s
600:	learn: 0.9556651	test: 0.9132539	best: 0.9132755 (597)	total: 2m 54s	remaining: 1m 55s
700:	learn: 0.9600796	test: 0.9149819	best: 0.9149819 (700)	total: 3m 25s	remaining: 1m 27s
800:	learn: 0.9638981	test: 0.9152762	best: 0.9153768 (773)	total: 4m 1s	remaining: 59.9s
900:	learn: 0.9667620	test: 0.9164957	best: 0.9165848 (886)	total: 4m 33s	remaining: 30s
999:	learn: 0.9695451	test: 0.9171226	best: 0.9171587 (986)	total: 5m 10s	remaining: 0us

be

## Chapter 6 — Model Comparison

Compare all three models side by side and pick the best for threshold optimisation.

In [26]:
from sklearn.metrics import recall_score


results = {}

for name, model in [('XGBoost', xgb_model), ('LightGBM', lgbm_model), ('CatBoost', cat_model)]:
    y_prob = model.predict_proba(X_val)[:, 1]
    results[name] = {
        'AUC-PR':  round(average_precision_score(y_val, y_prob), 4),
        'AUC-ROC': round(roc_auc_score(y_val, y_prob), 4),
        'F1':      round(f1_score(y_val, (y_prob >= 0.5).astype(int)), 4),
        'Recall':  round(recall_score(y_val, y_pred), 4)
    }

results_df = pd.DataFrame(results).T.sort_values('AUC-PR', ascending=False)
print(results_df.sort_values('AUC-PR', ascending=False))

best_model_name = results_df['AUC-PR'].head(1)
print(f'\nBest model: {best_model_name}')

          AUC-PR  AUC-ROC      F1  Recall
LightGBM  0.5265   0.9057  0.3784  0.7073
XGBoost   0.5231   0.9017  0.4017  0.7073
CatBoost  0.5133   0.9095  0.3684  0.7073

Best model: LightGBM    0.5265
Name: AUC-PR, dtype: float64
